#### Setup

In [1]:
import glob
import hashlib
import json
import os
import re

import numpy as np
import pandas as pd

In [2]:
DATA_DIR = "dataset"
OUT_DIR = os.path.join(DATA_DIR, "processed")  # inside dataset/, which .gitignore excludes
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_MONTHS = [str(p) for p in pd.period_range("2024-01", "2026-02", freq="M")]  # 26 months
VAL_MONTH = "2026-03"
TEST_MONTH = "2026-04"

# Later for step 2
MIN_SQFT_PER_BEDROOM = 70   # minimum habitable room size 
MAX_BATHS_PER_BEDROOM = 4   # above this = likely entry/parsing error (e.g. 175 for 1.75)

In [3]:
def month_file_map(data_dir=DATA_DIR):
    # 'YYYY-MM' -> CSV path, 
    # easier for pd.to_datetime to parse
    out = {}
    for f in glob.glob(os.path.join(data_dir, "CRMLSSold*.csv")):
        ym = re.search(r"CRMLSSold(\d{6})", os.path.basename(f)).group(1)
        out[f"{ym[:4]}-{ym[4:]}"] = f
    return dict(sorted(out.items()))


files = month_file_map()

#### Removal log
`drop_where` removes rows and records how many and why. 

`note` records corrections and flags that don't remove rows.

In [4]:
removal_log = []

def log_step(split, step, before, after, reason, action):
    # split: name of the dataset a row of the log belongs to: train, val, or test
    # step: step number for pre-processing plan
    # rows_before: the number of rows in the dataframe going into that step
    # rows_after: the number of rows left after that step
    # removed: number of rows that step dropped
    # percent_removed: the percent of rows that one step removed out of the rows that went into that step
    # percent_removed is good best practice
    removal_log.append({
        "split": split, "pre_processing_step": step, "rows_before": before, "rows_after": after,
        "removed": before - after,
        "percent_removed": round(100 * (before - after) / before, 3) if before else 0.0,
        "action": action, "reason": reason,
    })


def drop_where(df, mask, split, step, reason):
    # Drop rows where mask is True, and log it. Returns a new dataframe.
    before = len(df)
    out = df.loc[~mask].copy()
    log_step(split, step, before, len(out), reason, "dropped")
    return out


def note(df, split, step, reason, action="corrected"):
     # Log a correction or flag (no rows removed).
    log_step(split, step, len(df), len(df), reason, action)

#### Step 1: Load and filter

In [5]:
def load_months(months, split):
    dfs = []
    for m in months:
        d = pd.read_csv(files[m], low_memory=False)
        d["source_month"] = m
        dfs.append(d)
     # aligns columns across months due to schema drift
     #  missing ones become NaN
    df = pd.concat(dfs, ignore_index=True) 
    log_step(split, "step_0_load", len(df), len(df), f"raw rows from {len(months)} file(s)", "loaded")
    return df

In [6]:
def to_dates(df):
    # Getting columns like CloseDate and ListingContractDate 
    # into a date object to work with
    df = df.copy()
    for c in ["CloseDate", "ListingContractDate"]:
        df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

In [7]:
# single family residence only
def filter_scope(df, split):
    sfr = (df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")
    df = drop_where(df, ~sfr, split, "step_1_scope_sfr", "not Residential / SingleFamilyResidence (task doc)")
    df = drop_where(df, df["StateOrProvince"] != "CA", split, "step_1_scope_ca", "StateOrProvince is not CA")
    return df

#### Step 2: Remove records that can't be true
These rules use no learned values, so each split is cleaned on its own.

In [8]:
def normalize_address(s):
    # lowercase, trim, collapse spaces
    # useful for: '123  Main St ' and '123 main st' match
    return s.astype("string").str.lower().str.strip().str.replace(r"\s+", " ", regex=True)


def remove_impossible(df, split):
    
    # bad values such as NaN or negative
    df = drop_where(df, df["ClosePrice"].isna() | (df["ClosePrice"] <= 0),
                    split, "step_2_target", "ClosePrice missing or <= 0 (never impute the target)")
    
    
    # If the living area is smaller than that total, the record can't be true.
    # For example, given each bedroom at least 70 sq ft and each bathroom at least about 35 sq ft 
    # Then a two bed 1 bath should be >= 140 + 35 living area >= total
    df = drop_where(df, df["LivingArea"].isna() | (df["LivingArea"] <= 0),
                    split, "step_2_living_area", "LivingArea missing or <= 0")

    # Assumptions about the homes
    beds = df["BedroomsTotal"]
    MIN_SQFT_PER_BEDROOM = 70
    MAX_BATHS_PER_BEDROOM = 4

    beds = df["BedroomsTotal"]

    # 175 may be 1.75
    # No full, half, 3/4 bath to correct from? set to missing and keep the record
    # Step 4 imputes the train median and adds BathroomsTotalInteger_missing
    baths_per_bed = df["BathroomsTotalInteger"] / beds.clip(lower=1) # 0 beds counts as 1
    bad_baths = baths_per_bed > MAX_BATHS_PER_BEDROOM
    df.loc[bad_baths, "BathroomsTotalInteger"] = np.nan
    note(df, split, "step_2_baths_set_missing",
         f"set {int(bad_baths.sum()):,} BathroomsTotalInteger to missing "
         f"(> {MAX_BATHS_PER_BEDROOM} per bedroom; likely entry error, not correctable)")

    # bedrooms can't fit.
    # e.g. based on assumption bedroom means 4*70 = 280 sqft so 100 sqft won't work
    # needed for price/sqft and comp so remove the bad record
    too_small = beds.gt(0) & (df["LivingArea"] / beds < MIN_SQFT_PER_BEDROOM)
    df = drop_where(df, too_small, split, "step_2_beds_vs_area",
                    f"LivingArea < {MIN_SQFT_PER_BEDROOM} sqft per bedroom (area likely wrong)")
    
    # dates that don't make sense
    bad_dates = df["CloseDate"] < df["ListingContractDate"]
    df = drop_where(df, bad_dates, split, "step_2_close_before_list", "CloseDate earlier than ListingContractDate")

    # Only drop WRONG coordinates. 
    # Missing ones are filled median latitude and longitude 
    # of the home's city, computed from train in step 4
    # Why? Feb 2024 is missing ~25% 
    lat, lon = df["Latitude"], df["Longitude"]
    bad_geo = lat.notna() & lon.notna() & (
        (lat == 0) | (lon == 0) | ~lat.between(32.5, 42.1) | ~lon.between(-124.5, -114.1))
    df = drop_where(df, bad_geo, split, "step_2_geo", "lat/long is 0 or outside California")

    # Duplicates: same address + CloseDate = same sale. 
    # A different CloseDate is a real resale so we keep it.
    # Matters for comparisons (comps): a duplicate would become a 'perfect comp' for itself.
    df["address_norm"] = normalize_address(df["UnparsedAddress"])
    dup = df["address_norm"].notna() & df.duplicated(subset=["address_norm", "CloseDate"], keep="last")
    df = drop_where(df, dup, split, "step_2_duplicates", "duplicate sale (same address + CloseDate)")
    return df

#### Step 3: Outliers (thresholds learned from train)
`fit_params` learns every data-derived value from train. It's saved to `params.json` so val/test use the exact same numbers.

In [9]:
NUM_IMPUTE = ["LotSizeSquareFeet", "GarageSpaces", "YearBuilt", "BedroomsTotal", "BathroomsTotalInteger"]
CAT_MODE = ["CountyOrParish"]
CAT_UNKNOWN = ["Levels", "City"]  # 'Unknown' instead of the mode. Note Levels had ~9% missing.
YN_COLS = ["ViewYN", "PoolPrivateYN", "FireplaceYN", "NewConstructionYN"]


def fit_params(train):
    p = {}
    p["price_lo"], p["price_hi"] = train["ClosePrice"].quantile([0.005, 0.995]).tolist()
    ppsf = train["ClosePrice"] / train["LivingArea"]
    p["ppsf_lo"], p["ppsf_hi"] = ppsf.quantile([0.005, 0.995]).tolist()
    p["num_median"] = {c: float(train[c].median()) for c in NUM_IMPUTE}
    p["cat_mode"] = {c: str(train[c].mode().iloc[0]) for c in CAT_MODE}
    cent = train.groupby("City")[["Latitude", "Longitude"]].median().dropna()
    p["city_centroid"] = {str(k): [float(a), float(b)] for k, (a, b) in cent.iterrows()}
    p["global_centroid"] = [float(train["Latitude"].median()), float(train["Longitude"].median())]
    return p


def remove_outliers(df, p, split):
    df = drop_where(df, ~df["ClosePrice"].between(p["price_lo"], p["price_hi"]),
                    split, "step_3_price_outlier",
                    f"ClosePrice outside train 0.5th-99.5th percent (${p['price_lo']:,.0f}-${p['price_hi']:,.0f})")
    # ppsf: price per square foot
    ppsf = df["ClosePrice"] / df["LivingArea"]
    df = drop_where(df, ~ppsf.between(p["ppsf_lo"], p["ppsf_hi"]),
                    split, "step_3_ppsf_outlier",
                    f"price/sqft outside train 0.5th-99.5th percent (${p['ppsf_lo']:,.0f}-${p['ppsf_hi']:,.0f})")

    # Inspect only 
    # ListPrice is for cleaning, not for feature
    ratio = df["ClosePrice"] / df["ListPrice"].where(df["ListPrice"] > 0)
    df["flag_sale_to_list"] = ((ratio > 2) | (ratio < 0.5)).astype(int)
    note(df, split, "step_3_sale_to_list_flag",
         f"flagged {int(df['flag_sale_to_list'].sum())} rows with ClosePrice/ListPrice > 2 or < 0.5 (kept, inspect)",
         action="flagged")
    return df

#### Step 4: Missing values (values learned from train)

In [10]:
def fill_missing(df, p, split):
    df = df.copy()
    for c, v in p["num_median"].items():
        # missingness can be informative (handbook section 6)
        # number of missing values
        n = int(df[c].isna().sum())
        df[f"{c}_missing"] = df[c].isna().astype(int) 
        df[c] = df[c].fillna(v)
        note(df, split, f"step_4_impute_{c}", f"filled {n:,} missing with train median {v:g}")
    for c, v in p["cat_mode"].items():
        n = int(df[c].isna().sum())
        df[c] = df[c].fillna(v)
        note(df, split, f"step_4_impute_{c}", f"filled {n:,} missing with train mode '{v}'")
    for c in CAT_UNKNOWN:
        n = int(df[c].isna().sum())
        df[c] = df[c].fillna("Unknown")
        note(df, split, f"step_4_impute_{c}", f"filled {n:,} missing with 'Unknown'")


    # Missing lat/long -> train city centroid and fill with overall train median.
    # Keep latlon_missing: these homes must stay OUT of the comparison pool bc
    # their location is a city average, which isn't actually at that spot.
    # In week 6-7, we're going to build comparison features
    df["latlon_missing"] = (df["Latitude"].isna() | df["Longitude"].isna()).astype(int)
    fill = df["City"].map(p["city_centroid"])
    fill = fill.where(fill.notna(), pd.Series([p["global_centroid"]] * len(df), index=df.index))
    df["Latitude"] = df["Latitude"].fillna(fill.str[0])
    df["Longitude"] = df["Longitude"].fillna(fill.str[1])
    note(df, split, "step_4_impute_latlon", f"filled {int(df['latlon_missing'].sum()):,} missing lat/long with train city centroid")

    # AssociationFee: the data uses 0 for "no HOA", so MISSING = unknown, not "no HOA"
    # (missing-fee homes have a much higher median price: $1.2M vs $765k for $0 fee).
    # Keep the rows (~30% of data) and flag them. No frequency given -> assume monthly.
    # AssociationFee is reported at different frequencies convert everything to monthly
    FEE_TO_MONTHLY = {"Monthly": 1, "Annually": 1 / 12, "Quarterly": 1 / 3, "SemiAnnually": 1 / 6}

    freq = df["AssociationFeeFrequency"] if "AssociationFeeFrequency" in df else pd.Series(pd.NA, index=df.index)
    unknown = freq.notna() & ~freq.isin(list(FEE_TO_MONTHLY))
    if unknown.any():
        print(f"[{split}] unmapped AssociationFeeFrequency values: {freq[unknown].unique()}")
    n_converted = int((freq.notna() & (freq != "Monthly")).sum())
    monthly_fee = df["AssociationFee"] * freq.map(FEE_TO_MONTHLY).fillna(1)

    n = int(df["AssociationFee"].isna().sum())
    df["AssociationFee_missing"] = df["AssociationFee"].isna().astype(int)
    df["AssociationFee"] = monthly_fee.fillna(0)
    df["HasHOA"] = (df["AssociationFee"] > 0).astype(int)
    note(df, split, "4_impute_AssociationFee",
            f"converted {n_converted:,} non-monthly fees to monthly; filled {n:,} missing with 0 "
            f"+ AssociationFee_missing flag; HasHOA = fee > 0")

    for c in YN_COLS:
        df[f"{c}_missing"] = df[c].isna().astype(int)
        df[c] = df[c].map({True: 1, False: 0, "True": 1, "False": 0}).fillna(0).astype(int)
    return df

#### Steps 5–6: Drop columns + light features
Leakage list follows handbook §04. Target encoding, one-hot, log transforms and scaling wait for the Week 4 Pipeline (fit inside CV folds).

In [11]:
LEAKAGE = ["ListPrice", "OriginalListPrice", "DaysOnMarket", "PurchaseContractDate", "ContractStatusChangeDate", "flag_sale_to_list"]

#cols = IDS + ["ClosePrice"] + MODEL_FEATURE_1 + MODEL_FEATURE_2 + ... + MODEL_FEATURE_N + flags (all missing columns) + COMPS_ONLY )(week 6-7)

# Agent/office names, emails and phones aren't listed here
# they're excluded because they're not in the allowed.
# The assert below only checks this list.
PII = ["UnparsedAddress", "address_norm"]  

MODEL_FEATURES = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
    "GarageSpaces", "Levels", "AssociationFee", "HasHOA",
    "Latitude", "Longitude", "City", "CountyOrParish", "PostalCode", *YN_COLS,
]
# Kept for comparisions for weeks 6-7, but NOT model features on their own
COMPS_ONLY = ["CloseDate", "latlon_missing", "property_id", "price_per_sqft"]
IDS = ["source_month", "ListingId"]


def finalize(df, split):
    df = df.copy()
    # property_id: links resales of the same house without keeping the address (PII)
    df["property_id"] = df["address_norm"].map(
        lambda a: hashlib.sha256(a.encode()).hexdigest()[:16] if isinstance(a, str) else pd.NA)
    df["price_per_sqft"] = df["ClosePrice"] / df["LivingArea"]  # for summarizing NEIGHBORS only, and contains the target

    df["AgeAtSale"] = df["CloseDate"].dt.year - df["YearBuilt"]
    month = df["CloseDate"].dt.month
    df["month_sin"] = np.sin(2 * np.pi * month / 12)  # cyclical: December and January stay neighbors
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    flags = [c for c in df.columns if c.endswith("_missing") and c != "latlon_missing"]
    cols = IDS + ["ClosePrice"] + MODEL_FEATURES + ["AgeAtSale", "month_sin", "month_cos"] + flags + COMPS_ONLY
    assert not set(cols) & set(LEAKAGE + PII), "leakage/PII column slipped into the output"
    n_dropped_cols = df.shape[1] - len(cols)
    out = df[cols]
    note(out, split, "step_5_drop_columns",
         f"kept {len(cols)} columns, dropped {n_dropped_cols} (leakage, PII, empty, redundant)", action="columns")
    return out

#### Run: same logic on all three splits

In [12]:
def clean_rows(months, split):
    df = load_months(months, split)
    df = to_dates(df)
    df = filter_scope(df, split)
    df = remove_impossible(df, split)
    return df


stage = {
    "train": clean_rows(TRAIN_MONTHS, "train"),
    "val": clean_rows([VAL_MONTH], "val"),
    "test": clean_rows([TEST_MONTH], "test"),
}

params = fit_params(stage["train"])  # TRAIN ONLY
params["cleaning_rules"] = {
    "min_sqft_per_bedroom": MIN_SQFT_PER_BEDROOM,
    "max_baths_per_bedroom": MAX_BATHS_PER_BEDROOM,
}
params["data_snapshot"] = {
    "files": {m: os.path.basename(files[m]) for m in [*TRAIN_MONTHS, VAL_MONTH, TEST_MONTH]},
    "raw_rows": {r["split"]: r["rows_before"] for r in removal_log if r["pre_processing_step"] == "step_0_load"},
    "pandas_version": pd.__version__,
}
with open(os.path.join(OUT_DIR, "params.json"), "w") as f:
    json.dump(params, f, indent=2)
print({k: v for k, v in params.items() if k != "city_centroid"})

clean = {}
for split, df in stage.items():
    df = remove_outliers(df, params, split)
    df = fill_missing(df, params, split)
    clean[split] = finalize(df, split)

{k: v.shape for k, v in clean.items()}

{'price_lo': 190000.0, 'price_hi': 8200000.0, 'ppsf_lo': 145.43439159172175, 'ppsf_hi': 2283.087189551251, 'num_median': {'LotSizeSquareFeet': 7243.0, 'GarageSpaces': 2.0, 'YearBuilt': 1976.0, 'BedroomsTotal': 3.0, 'BathroomsTotalInteger': 2.0}, 'cat_mode': {'CountyOrParish': 'Los Angeles'}, 'global_centroid': [34.09195, -118.04127265], 'cleaning_rules': {'min_sqft_per_bedroom': 70, 'max_baths_per_bedroom': 4}, 'data_snapshot': {'files': {'2024-01': 'CRMLSSold202401_filled.csv', '2024-02': 'CRMLSSold202402.csv', '2024-03': 'CRMLSSold202403_filled.csv', '2024-04': 'CRMLSSold202404_filled.csv', '2024-05': 'CRMLSSold202405_filled.csv', '2024-06': 'CRMLSSold202406_filled.csv', '2024-07': 'CRMLSSold202407_filled.csv', '2024-08': 'CRMLSSold202408.csv', '2024-09': 'CRMLSSold202409.csv', '2024-10': 'CRMLSSold202410.csv', '2024-11': 'CRMLSSold202411.csv', '2024-12': 'CRMLSSold202412.csv', '2025-01': 'CRMLSSold202501_filled.csv', '2025-02': 'CRMLSSold202502.csv', '2025-03': 'CRMLSSold202503.csv'

{'train': (280711, 38), 'val': (11347, 38), 'test': (12238, 38)}

#### Step 7: Save and Summary

In [13]:
for split, df in clean.items():
    df.to_csv(os.path.join(OUT_DIR, f"{split}_clean.csv"), index=False)

log = pd.DataFrame(removal_log)
log.to_csv(os.path.join(OUT_DIR, "removal_log.csv"), index=False)

# Row counts before and after cleaning, per dataset
counts = pd.DataFrame({
    "raw_rows": log[log["pre_processing_step"] == "step_0_load"].set_index("split")["rows_before"],
    "clean_rows": pd.Series({k: len(v) for k, v in clean.items()}),
})
counts["kept_percent"] = (100 * counts["clean_rows"] / counts["raw_rows"]).round(1)
counts

,raw_rows,clean_rows,kept_percent
train,568074,280711,49.4
val,23372,11347,48.5
test,24261,12238,50.4


In [14]:
# Rows removed at each step: count and % (of rows entering that step), per split
summary = (log[log["action"] == "dropped"]
           .pivot_table(index="pre_processing_step", columns="split", values=["removed", "percent_removed"],
                        aggfunc="first", sort=False)
           .swaplevel(axis=1)  # group columns by split instead of by value
           .reindex(columns=pd.MultiIndex.from_product([["train", "val", "test"], ["removed", "percent_removed"]])))
summary

train                     val                  \
                         removed percent_removed removed percent_removed   
pre_processing_step                                                        
step_1_scope_sfr          282391          49.710   11810          50.531   
step_1_scope_ca               21           0.007       0           0.000   
step_2_target                  3           0.001       0           0.000   
step_2_living_area           276           0.097       4           0.035   
step_2_beds_vs_area            5           0.002       0           0.000   
step_2_close_before_list      40           0.014       1           0.009   
step_2_geo                    57           0.020       1           0.009   
step_2_duplicates            173           0.061       3           0.026   
step_3_price_outlier        2807           0.985     140           1.212   
step_3_ppsf_outlier         1590           0.563      66           0.578   

                            test                  
                         removed percent_removed  
pre_processing_step                               
step_1_scope_sfr           11771          48.518  
step_1_scope_ca                1           0.008  
step_2_target                  0           0.000  
step_2_living_area            10           0.080  
step_2_beds_vs_area            0           0.000  
step_2_close_before_list       3           0.024  
step_2_geo                     4           0.032  
step_2_duplicates              8           0.064  
step_3_price_outlier         138           1.107  
step_3_ppsf_outlier           88           0.714

In [15]:
# Corrections, flags and column drops, with full reasons (e.g. imputation values)
pd.set_option("display.max_colwidth", None)
log[log["action"].isin(["corrected", "flagged", "columns"])][["split", "pre_processing_step", "reason"]]

,split,pre_processing_step,reason
5,train,step_2_baths_set_missing,"set 8 BathroomsTotalInteger to missing (> 4 per bedroom; likely entry error, not correctable)"
15,val,step_2_baths_set_missing,"set 0 BathroomsTotalInteger to missing (> 4 per bedroom; likely entry error, not correctable)"
25,test,step_2_baths_set_missing,"set 0 BathroomsTotalInteger to missing (> 4 per bedroom; likely entry error, not correctable)"
32,train,step_3_sale_to_list_flag,"flagged 46 rows with ClosePrice/ListPrice > 2 or < 0.5 (kept, inspect)"
33,train,step_4_impute_LotSizeSquareFeet,"filled 4,838 missing with train median 7243"
34,train,step_4_impute_GarageSpaces,"filled 10,045 missing with train median 2"
35,train,step_4_impute_YearBuilt,filled 136 missing with train median 1976
36,train,step_4_impute_BedroomsTotal,filled 0 missing with train median 3
37,train,step_4_impute_BathroomsTotalInteger,filled 52 missing with train median 2
38,train,step_4_impute_CountyOrParish,filled 0 missing with train mode 'Los Angeles'


Use the template from earlier: the summary bullets, leakage audit table, assumptions, limitations, and next steps. Make two changes to it:
- Assumptions: "A missing AssociationFee is unknown, not 'no HOA'". Delete the "Verify missing AssociationFee = no HOA" TODO, since that's been checked now.
- Limitations: "Bathroom counts > 4 per bedroom are set to missing (likely entry errors such as 175 = 1.75; no full/half bath fields exist to correct them)." 
Also:
- "MainLevelBedrooms > BedroomsTotal in 1,744 train rows: which field do we trust?"